## Let's first create a Noise model with Aer Simulator

In [2]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, QuantumError, pauli_error

In [3]:
def my_noise_model(p_x: float, p_z: float) -> NoiseModel:
    assert p_x >= 0 and p_z >= 0 and p_x + p_z <= 1, "Probabilities must be non-negative and sum to at most 1."

    # Define the Pauli error channel
    error_channel = pauli_error([
        ('X', p_x),           # bit-flip
        ('Z', p_z),           # phase-flip
        ('I', 1 - p_x - p_z) # no error (identity)
    ])

    noise_model = NoiseModel()

    # Apply to all single-qubit gates
    single_qubit_gates = ['h', 'x', 'y', 'z', 's', 't', 'rx', 'ry', 'rz', 'u']
    noise_model.add_all_qubit_quantum_error(error_channel, single_qubit_gates)

    # For 2-qubit gates, tensor two independent single-qubit channels
    two_qubit_error = error_channel.tensor(error_channel)
    noise_model.add_all_qubit_quantum_error(two_qubit_error, ['cx', 'cz', 'swap'])

    return noise_model


In [4]:
def simulate_circuit_with_noise(qc: QuantumCircuit, p_x: float, p_z: float, shots: int = 1024):
    noise_model = my_noise_model(p_x, p_z)
    simulator = AerSimulator(noise_model=noise_model)

    #If measure is not in the circuit
    if not qc.cregs:
        qc_measured = qc.copy()
        qc_measured.measure_all()
    else:
        qc_measured = qc

    result = simulator.run(qc_measured, shots=shots).result()
    return result.get_counts()

In [5]:
# Bell state — should be ~50/50 '00'/'11' without noise,
# but errors will bleed in counts like '01', '10'
def n_ghz(num_qubits):
    n_ghz = QuantumCircuit(num_qubits)
    n_ghz.h(0)
    for i in range(1,num_qubits):
        n_ghz.cx(0,i)
    return n_ghz

ghz_5 = n_ghz(5)

counts = simulate_circuit_with_noise(ghz_5, p_x=0.05, p_z=0.01)


Let's find the leakage percentage happening

In [6]:
def leakage_percent(counts: dict, qc: QuantumCircuit) -> float:
    total_counts = sum(counts.values())
    if total_counts == 0:
        return 0.0
    
    no_noise = simulate_circuit_with_noise(qc, 0, 0, shots=total_counts)

    noisy_counts = counts

    # Find the expected outcomes 
    valid_keys = set(no_noise.keys())

    # get the sum of their probability
    ideal_counts = sum(noisy_counts[k] for k in noisy_counts if k in valid_keys)

    #find the leakage counts
    leakage_counts = total_counts - ideal_counts
    return (leakage_counts / total_counts) * 100

In [8]:
leakage_list = []
for i in range(50):
    counts = simulate_circuit_with_noise(ghz_5, p_x=0.05, p_z=0.01)
    leakage_list.append(leakage_percent(counts, ghz_5))


avg_leakage = sum(leakage_list) / len(leakage_list)
print(f"Average leakage percentage over 50 runs: {avg_leakage:.2f}%")


Average leakage percentage over 50 runs: 33.57%
